# WRDS API Exploration: `compd.wrds_segmerged`

In [2]:
from __future__ import annotations

import os
from getpass import getpass
from typing import Any
from urllib.parse import urljoin
import wrds
import pandas as pd
import requests
import matplotlib.pyplot as plt
import subprocess

In [3]:
# find repo root git
def find_repo_root(path: str) -> str:
    if os.path.isdir(os.path.join(path, '.git')):
        return path
    parent = os.path.dirname(path)
    if parent == path:
        raise FileNotFoundError("No .git directory found in any parent directories.")
    return find_repo_root(parent)

repo_root = find_repo_root(os.getcwd())
print(f"Repo root found at: {repo_root}")


# set up data (processed) r
data_path = os.path.join(repo_root, 'data', 'raw', 'wrds')
external_data_path = os.path.join(repo_root, 'data', 'external')
processed_data_path = os.path.join(repo_root, 'data', 'processed', 'wrds')
os.makedirs(data_path, exist_ok=True)
os.makedirs(processed_data_path, exist_ok=True)



Repo root found at: /Users/hzf/Library/CloudStorage/Dropbox/Github/bayes_market_external


## 1. Configure token-auth session

In [4]:
conn = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
conn.list_libraries().sort()
type(conn.list_libraries())

list

In [6]:
conn.list_tables(library='comp_na_daily_all')

['aco_amda',
 'aco_imda',
 'aco_indfnta',
 'aco_indfntq',
 'aco_indfntytd',
 'aco_indsta',
 'aco_indstq',
 'aco_indstytd',
 'aco_notesa',
 'aco_notesq',
 'aco_notessa',
 'aco_notesytd',
 'aco_pnfnda',
 'aco_pnfndq',
 'aco_pnfndytd',
 'aco_pnfnta',
 'aco_pnfntq',
 'aco_pnfntytd',
 'aco_transa',
 'aco_transq',
 'aco_transsa',
 'aco_transytd',
 'adsprate',
 'asec_amda',
 'asec_imda',
 'asec_notesa',
 'asec_notesq',
 'asec_transa',
 'asec_transq',
 'chars',
 'co_aacctchg',
 'co_aaudit',
 'co_acthist',
 'co_adesind',
 'co_adjfact',
 'co_afnd1',
 'co_afnd2',
 'co_afnddc1',
 'co_afnddc2',
 'co_afntind1',
 'co_afntind2',
 'co_ainvval',
 'co_amkt',
 'co_busdescl',
 'co_cotype',
 'co_filedate',
 'co_fortune',
 'co_hgic',
 'co_iacctchg',
 'co_iaudit',
 'co_idesind',
 'co_ifndq',
 'co_ifndsa',
 'co_ifndytd',
 'co_ifntq',
 'co_ifntsa',
 'co_ifntytd',
 'co_imkt',
 'co_industry',
 'co_ipcd',
 'co_mthly',
 'co_offtitl',
 'company',
 'currency',
 'dd_group',
 'dd_group_xref',
 'dd_item',
 'dd_package',

In [7]:
conn.list_tables(library='comp')

['aco_amda',
 'aco_imda',
 'aco_indfnta',
 'aco_indfntq',
 'aco_indfntytd',
 'aco_indsta',
 'aco_indstq',
 'aco_indstytd',
 'aco_notesa',
 'aco_notesq',
 'aco_notessa',
 'aco_notesytd',
 'aco_pnfnda',
 'aco_pnfndq',
 'aco_pnfndytd',
 'aco_pnfnta',
 'aco_pnfntq',
 'aco_pnfntytd',
 'aco_transa',
 'aco_transq',
 'aco_transsa',
 'aco_transytd',
 'adsprate',
 'asec_amda',
 'asec_imda',
 'asec_notesa',
 'asec_notesq',
 'asec_transa',
 'asec_transq',
 'bank_aacctchg',
 'bank_adesind',
 'bank_afnd1',
 'bank_afnd2',
 'bank_afnddc1',
 'bank_afnddc2',
 'bank_afntind',
 'bank_funda',
 'bank_funda_fncd',
 'bank_fundq',
 'bank_fundq_fncd',
 'bank_iacctchg',
 'bank_idesind',
 'bank_ifndq',
 'bank_ifndytd',
 'bank_ifntq',
 'bank_ifntytd',
 'bank_names',
 'bank_namesq',
 'chars',
 'co_aacctchg',
 'co_aaudit',
 'co_acthist',
 'co_adesind',
 'co_adjfact',
 'co_afnd1',
 'co_afnd2',
 'co_afnddc1',
 'co_afnddc2',
 'co_afntind1',
 'co_afntind2',
 'co_ainvval',
 'co_amkt',
 'co_busdescl',
 'co_cotype',
 'co_f

In [8]:
company_name = conn.get_table(library='comp', table='company')
company_name.shape

(57397, 40)

In [9]:
# i need conm, gvkey, cik, city, conml, curr_sp500_flag
company_name = company_name[['conm', 'gvkey', 'cik', 'city', 'conml', 'curr_sp500_flag']]
company_name.head()

,conm,gvkey,cik,city,conml,curr_sp500_flag
0,A & E PLASTIK PAK INC,001000,<NA>,<NA>,A & E Plastik Pak Inc,0.0
1,A & M FOOD SERVICES INC,001001,0000723576,Tulsa,A & M Food Services Inc,0.0
2,AAI CORP,001002,0001306124,Hunt Valley,AAI Corp,0.0
3,A.A. IMPORTING CO INC,001003,0000730052,St. Louis,A.A. Importing Co Inc,0.0
4,AAR CORP,001004,0000001750,Wood Dale,AAR Corp,0.0


In [10]:

company = conn.get_table(library='comp_na_daily_all', table='wrds_segmerged')
company.shape

/opt/anaconda3/envs/bayes_ml/lib/python3.11/site-packages/wrds/sql.py:604: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_df = pd.concat([full_df, chunk])


(511378, 92)

In [11]:
company.columns 

# merge company with company_name on gvkey
company_merged = pd.merge(company, company_name, on='gvkey', how='left')
company_merged.shape

(511378, 97)

In [12]:
company_merged.columns

Index(['gvkey', 'stype', 'sid', 'atlls', 'atlls_dc', 'capxs', 'capxs_dc',
       'capxs_fn', 'caxts', 'caxts_dc', 'cogss', 'cogss_dc', 'dps', 'dps_dc',
       'emps', 'emps_dc', 'emps_fn', 'esubs', 'esubs_dc', 'esubs_fn', 'gdwls',
       'gdwls_dc', 'ias', 'ias_dc', 'ibs', 'ibs_dc', 'iints', 'iints_dc',
       'intseg', 'intseg_dc', 'ivaeqs', 'ivaeqs_dc', 'nis', 'nis_dc', 'nopxs',
       'nopxs_dc', 'nxints', 'nxints_dc', 'obs', 'obs_dc', 'ocaxs', 'ocaxs_dc',
       'oelim', 'oelim_dc', 'oiadps', 'oiadps_dc', 'oibdps', 'oibdps_dc',
       'ops', 'ops_dc', 'ops_fn', 'ppents', 'ppents_dc', 'ptis', 'ptis_dc',
       'rds', 'rds_dc', 'rds_fn', 'revts', 'revts_dc', 'sales', 'sales_dc',
       'sales_fn', 'salexg', 'salexg_dc', 'spis', 'spis_dc', 'txts', 'txts_dc',
       'txws', 'txws_dc', 'xidos', 'xidos_dc', 'xints', 'xints_dc', 'xsgas',
       'xsgas_dc', 'datadate', 'srcdate', 'curcds', 'isosrc', 'naicsh', 'srcs',
       'upds', 'naicss1', 'naicss2', 'sics1', 'sics2', 'geotp', 'snms',
 

In [13]:
# conm, gvkey, cik, scrdate,  datadate, curcds, naicsh, naicss1, naicss2, snms, geotp, soptp1, curr_sp500_flag
company_merged = company_merged[['conm', 'gvkey', 'cik', 'srcdate', 'datadate', 'curcds', 'naicsh', 'naicss1', 'naicss2', 'snms', 'geotp', 'soptp1', 'curr_sp500_flag', 'sales', 'sales_dc', 'sales_fn']]

In [14]:
company_merged

,conm,gvkey,cik,srcdate,datadate,curcds,naicsh,naicss1,naicss2,snms,geotp,soptp1,curr_sp500_flag,sales,sales_dc,sales_fn
0,AAR CORP,001004,0000001750,2020-05-31,2018-05-31,USD,423860,<NA>,<NA>,Corporate,<NA>,PD_SRVC,0.0,0.0,<NA>,<NA>
1,AAR CORP,001004,0000001750,2020-05-31,2018-05-31,USD,423860,423860,488190,Aviation Services,<NA>,PD_SRVC,0.0,1635.8,<NA>,<NA>
2,AAR CORP,001004,0000001750,2020-05-31,2018-05-31,USD,423860,481211,336413,Expeditionary Services,<NA>,PD_SRVC,0.0,112.5,<NA>,<NA>
3,AAR CORP,001004,0000001750,2020-05-31,2018-05-31,USD,423860,<NA>,<NA>,Other,3,GEO,0.0,178.8,<NA>,<NA>
4,AAR CORP,001004,0000001750,2020-05-31,2018-05-31,USD,423860,<NA>,<NA>,North America,2,GEO,0.0,1236.7,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511373,AGI INC,370994,0002081206,2024-12-31,2023-12-31,USD,522291,522291,522110,Cayman Islands,2,GEO,0.0,<NA>,8,<NA>
511374,AGI INC,370994,0002081206,2024-12-31,2023-12-31,USD,522291,522291,522110,Brazil,3,GEO,0.0,645.843,<NA>,<NA>
511375,AGI INC,370994,0002081206,2024-12-31,2024-12-31,USD,522291,522291,522110,Banking,<NA>,PD_SRVC,0.0,734.788,<NA>,<NA>
511376,AGI INC,370994,0002081206,2024-12-31,2024-12-31,USD,522291,522291,522110,Cayman Islands,2,GEO,0.0,<NA>,8,<NA>


In [15]:

# see if AMAZON is in conm (any contains AMAZON)
company_merged[company_merged['conm'].str.contains('AMAZON', case=False, na=False)]

,conm,gvkey,cik,srcdate,datadate,curcds,naicsh,naicss1,naicss2,snms,geotp,soptp1,curr_sp500_flag,sales,sales_dc,sales_fn
350622,AMAZON.COM INC,064768,0001018724,2019-12-31,2017-12-31,USD,454110,<NA>,<NA>,United States,2,GEO,1.0,120486.0,<NA>,<NA>
350623,AMAZON.COM INC,064768,0001018724,2019-12-31,2017-12-31,USD,454110,<NA>,<NA>,Rest of World,3,GEO,1.0,17150.0,<NA>,<NA>
350624,AMAZON.COM INC,064768,0001018724,2019-12-31,2017-12-31,USD,454110,<NA>,<NA>,Germany,3,GEO,1.0,16951.0,<NA>,<NA>
350625,AMAZON.COM INC,064768,0001018724,2019-12-31,2017-12-31,USD,454110,<NA>,<NA>,United Kingdom,3,GEO,1.0,11372.0,<NA>,<NA>
350626,AMAZON.COM INC,064768,0001018724,2019-12-31,2017-12-31,USD,454110,<NA>,<NA>,Japan,3,GEO,1.0,11907.0,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350806,AMAZON.COM INC,064768,0001018724,2025-12-31,2025-12-31,USD,455219,<NA>,<NA>,Japan,3,GEO,1.0,30688.0,<NA>,<NA>
350807,AMAZON.COM INC,064768,0001018724,2025-12-31,2025-12-31,USD,455219,455219,519290,North America,2,GEO,1.0,426305.0,<NA>,<NA>
350808,AMAZON.COM INC,064768,0001018724,2025-12-31,2025-12-31,USD,455219,455219,519290,International,3,GEO,1.0,161894.0,<NA>,<NA>
350809,AMAZON.COM INC,064768,0001018724,2025-12-31,2025-12-31,USD,455219,519290,541519,Amazon Web Services (AWS),<NA>,PD_SRVC,1.0,128725.0,<NA>,<NA>


# Clean WRDS data
## if you dont have a WRDS acct, start here by readin the parquet

In [16]:
# export this to a parquet file to data_path 
company_merged.to_parquet(f'{data_path}/comp_na_daily_all.parquet', index=False)
company_merged = pd.read_parquet(f'{data_path}/comp_na_daily_all.parquet')

In [17]:
company_merged.columns

Index(['conm', 'gvkey', 'cik', 'srcdate', 'datadate', 'curcds', 'naicsh',
       'naicss1', 'naicss2', 'snms', 'geotp', 'soptp1', 'curr_sp500_flag',
       'sales', 'sales_dc', 'sales_fn'],
      dtype='object')

In [18]:
# construct firm_segment_name by conm + snms
company_merged['firm_segment_name'] = company_merged['conm'] + ' - ' + company_merged['snms']

# construct year from datadate eg 2019-09-30 to 2019
company_merged['year'] = company_merged['datadate'].str[:4]



company_merged_dedup = company_merged.drop_duplicates(subset=['conm', 'gvkey', 'datadate', 'cik', 'srcdate', 'curcds', 'naicsh', 'naicss1', 'naicss2', 'snms', 'geotp', 'soptp1', 'curr_sp500_flag', 'year'])


# Aggregate sales by segment plus the metadata we need for domestic labeling.
df_agg = company_merged_dedup.groupby(['conm', 'firm_segment_name', 'gvkey', 'cik', 'srcdate', 'datadate', 'curcds', 'naicsh', 'naicss1', 'naicss2', 'snms', 'geotp', 'soptp1', 'curr_sp500_flag', 'year']).agg({'sales': 'sum', 'sales_dc': 'sum', 'sales_fn': 'sum'}).reset_index()

# Safe first-pass rule from WRDS segment geography coding:
# geotp == 2 means Domestic.
# Other observed values are treated as Non-Domestic for now.
# This avoids incorrectly equating Domestic with US.
df_agg['country'] = 'Non-Domestic'
df_agg.loc[df_agg['geotp'] == 2, 'country'] = 'Domestic'

# Separate heuristic for likely US-facing segment names.
# User decision: count "North America" as US for this exploratory flag.
segment_name_lower = df_agg['snms'].fillna('').str.lower()

us_name_mask = (
    segment_name_lower.str.contains(r'\bunited states\b', regex=True)
    | segment_name_lower.str.contains(r'\bu\.s\.\b', regex=True)
    | segment_name_lower.str.contains(r'\bu\.s\b', regex=True)
    | segment_name_lower.str.contains(r'\bus\b', regex=True)
    | segment_name_lower.str.contains(r'north america', regex=True)
)

non_us_name_mask = (
    segment_name_lower.str.contains(r'international', regex=True)
    | segment_name_lower.str.contains(r'foreign', regex=True)
    | segment_name_lower.str.contains(r'non-u\.s\.|non-us|non u\.s\.|non us', regex=True)
    | segment_name_lower.str.contains(r'europe|asia|china|japan|canada|latin america', regex=True)
)

df_agg['country_us_name'] = 'Ambiguous'
df_agg.loc[non_us_name_mask, 'country_us_name'] = 'Non-US'
df_agg.loc[us_name_mask, 'country_us_name'] = 'US'

df_agg_domestic = df_agg[df_agg['country'] == 'Domestic'].copy()
df_agg_non_domestic = df_agg[df_agg['country'] == 'Non-Domestic'].copy()
df_agg_us_name = df_agg[df_agg['country_us_name'] == 'US'].copy()

# Non-geographic segments are the ones most likely to need later manual interpretation.
is_geo_segment = df_agg['soptp1'].isin(['GEO', 'MARKET'])
df_pd = df_agg[~is_geo_segment].drop_duplicates(subset=['firm_segment_name'])[['firm_segment_name', 'conm', 'gvkey', 'geotp', 'country', 'country_us_name']]


In [19]:
df_agg_us_name 

,conm,firm_segment_name,gvkey,cik,srcdate,datadate,curcds,naicsh,naicss1,naicss2,snms,geotp,soptp1,curr_sp500_flag,year,sales,sales_dc,sales_fn,country,country_us_name
69,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2019-12-31,2017-12-31,USD,621112,621112,621999,United States,2,GEO,0.0,2017,176.769,0,0,Non-Domestic,US
70,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2019-12-31,2018-12-31,USD,621112,621112,621999,United States,2,GEO,0.0,2018,212.678,0,0,Non-Domestic,US
71,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2019-12-31,2019-12-31,USD,621112,621112,621999,United States,2,GEO,0.0,2019,276.258,0,0,Non-Domestic,US
72,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2020-12-31,2018-12-31,USD,621112,621112,621999,United States,2,GEO,0.0,2018,212.678,0,0,Non-Domestic,US
73,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2020-12-31,2019-12-31,USD,621112,621112,621999,United States,2,GEO,0.0,2019,276.258,0,0,Non-Domestic,US
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123714,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2020-12-31,2019-12-31,USD,511210,511210,5418,United States,2,MARKET,0.0,2019,826.556,0,0,Non-Domestic,US
123715,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2020-12-31,2020-12-31,USD,511210,511210,5418,United States,2,MARKET,0.0,2020,1211.7,0,0,Non-Domestic,US
123716,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2021-12-31,2019-12-31,USD,511210,511210,5418,United States,2,MARKET,0.0,2019,826.556,0,0,Non-Domestic,US
123717,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2021-12-31,2020-12-31,USD,511210,511210,5418,United States,2,MARKET,0.0,2020,1211.7,0,0,Non-Domestic,US


In [20]:
# i want a count of naics_code for each conm x firm_segment_name x gvkey x cik x curr_sp500_flag 
# eg
# firm,segment,year,naics_code,count
#AMZN,North America,2020,445110,5
#AMZN,North America,2021,452210,12
#AMZN,North America,2022,454110,8
#AMZN,North America,2023,445110,7
#AMZN,North America,2024,452210,10


# there are 3 columns telling us how can this segment be classified by naics code: naicsh, naicss1, naicss2. I want to count unique naics codes for each gvkey x firm x segment x year x 'curr_sp500_flag'
id_cols = [
    "conm",
    "firm_segment_name",
    "gvkey",
    "cik",
    "year",
    "curr_sp500_flag",
]

naics_cols = ["naicsh", "naicss1", "naicss2"]

df_naics_long = (
    df_agg_us_name
    .melt(
        id_vars=id_cols,
        value_vars=naics_cols,
        var_name="naics_source",
        value_name="naics_code"
    )
    .dropna(subset=["naics_code"])
)

# optional: clean NAICS as string
df_naics_long["naics_code"] = (
    df_naics_long["naics_code"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

df_naics_count = (
    df_naics_long
    .groupby(id_cols + ["naics_code"])
    .size()
    .reset_index(name="count")
    .sort_values(id_cols + ["count"], ascending=[True, True, True, True, True, True, False])
)

df_naics_count

,conm,firm_segment_name,gvkey,cik,year,curr_sp500_flag,naics_code,count
0,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2017,0.0,621112,2
1,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2017,0.0,621999,1
2,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2018,0.0,621112,4
3,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2018,0.0,621999,2
4,1LIFE HEALTHCARE INC,1LIFE HEALTHCARE INC - United States,036025,0001404123,2019,0.0,621112,6
...,...,...,...,...,...,...,...,...
41839,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2019,0.0,5418,3
41840,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2020,0.0,511210,4
41841,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2020,0.0,5418,2
41842,ZYNGA INC,ZYNGA INC - United States,187576,0001439404,2021,0.0,511210,2


In [21]:
# unique year in df_naics_count 17-26
df_naics_count['year'].unique()

# export df_naics_count to csv 
df_naics_count.to_csv(f'{processed_data_path}/wrds_naics_code_count_by_segmentxyear.csv', index=False)

# Clean Sales by Segment x year

In [22]:
company_merged_dedup = df_agg_us_name.drop_duplicates(subset=['firm_segment_name', 'year', 'conm', 'gvkey', 'cik', 'curcds', 'snms', 'geotp', 'soptp1', 'curr_sp500_flag'])

# aggregate sales by  conm, gvkey, cik, firm_segment_name, year, curr_sp500_flag
df_sales_agg_by_year = company_merged_dedup.groupby(['conm', 'firm_segment_name', 'gvkey', 'cik', 'year', 'curr_sp500_flag']).agg({'sales': 'sum', 'sales_dc': 'sum', 'sales_fn': 'sum'}).reset_index()

# drop sales_dc and sales_fn
df_sales_agg_by_year = df_sales_agg_by_year.drop(columns=['sales_dc', 'sales_fn'])
df_sales_agg_by_year 


# export to csv
df_sales_agg_by_year.to_csv(f'{processed_data_path}/wrds_sales_agg_by_segmentxyear.csv', index=False)

In [24]:
# Clean Sales by industry (naicsh) x year 
company_merged_dedup = df_agg_us_name.drop_duplicates(subset=['firm_segment_name', 'year', 'conm', 'gvkey', 'cik', 'curcds', 'snms', 'geotp', 'soptp1', 'curr_sp500_flag', 'naicsh'])

# Aggregate sales by  year, naicsh
df_sales_agg_by_naicsh_year = company_merged_dedup.groupby(['year', 'naicsh']).agg({'sales': 'sum'}).reset_index()


# read in raw 2017-NAICS-2-3-4-5-6-Digit-Codes-Listed-Numerically
#data_path
subprocess.run(['pip', 'install', 'openpyxl'], check=True)
naics_codes = pd.read_excel(f'{data_path}/2017-NAICS-2-3-4-5-6-Digit-Codes-Listed-Numerically.xlsx', dtype={'NAICS Code': str})

naics_codes.columns 
# rename columns 2017- All NAICS Codes: 2-6digit_naics 
# column '2017 NAICS Title (USA)' to 'naics_title'
naics_codes = naics_codes.rename(columns={
    'NAICS Code': 'naics_code',
    '2017 NAICS Title (USA)': 'naics_title'
})

# i only need these two columns
if "naics_code" not in naics_codes.columns and "2017- All NAICS Codes" in naics_codes.columns:
    naics_codes = naics_codes.rename(columns={"2017- All NAICS Codes": "naics_code"})



naics_codes = naics_codes[["naics_code", "naics_title"]]
naics_codes.head()

# add a year "2017" to naics_codes to make it easier for merging
naics_codes['year'] = '2017'


naics_code_2022 = pd.read_excel(f'{data_path}/2022-NAICS-Codes-listed-numerically-2-Digit-through-6-Digit.xlsx', dtype={'NAICS Code': str})

# rename columns 2022-NAICS-Codes-listed-numerically-2-Digit-through-6-Digit.xlsx
# '2022 NAICS US   Code' to 'naics_code', '2022 NAICS US Title' to 'naics_title'
naics_code_2022 = naics_code_2022.rename(columns={
    '2022 NAICS US   Code': 'naics_code',
    '2022 NAICS US Title': 'naics_title'
})
# i only need these two columns
# delete wierd T in naics_title eg 1780	62131	Offices of ChiropractorsT
naics_code_2022['naics_title'] = naics_code_2022['naics_title'].str.replace('T$', '', regex=True)
naics_code_2022

naics_code_2022 = naics_code_2022[["naics_code", "naics_title"]]

# add year column 2022 to naics_code_2022
naics_code_2022['year'] = '2022'

# concat naics_codes and naics_code_2022
naics_codes_17_22 = pd.concat([naics_codes, naics_code_2022], ignore_index=True)
naics_codes_17_22.head()

# dedup naics_codes_17_22 by naics_code  and naics_title
naics_codes_17_22 = naics_codes_17_22.drop_duplicates(subset=['naics_code', 'naics_title'])

# make sure they are string
naics_codes_17_22['naics_code'] = naics_codes_17_22['naics_code'].astype(str)
naics_codes_17_22['naics_title'] = naics_codes_17_22['naics_title'].astype(str)
# drop year column in naics_codes_17_22
naics_codes_17_22 = naics_codes_17_22.drop(columns=['year'])


# save naics_codes_17_22 to csv
naics_codes_17_22.to_csv(f'{processed_data_path}/naics_codes_17_22.csv', index=False)

# crosswalk naicsh in df_sales_agg_by_naicsh_year with naics_code in naics_codes to get naics_title, left join
df_sales_agg_by_naicsh_year = pd.merge(df_sales_agg_by_naicsh_year, naics_codes_17_22, left_on='naicsh', right_on='naics_code', how='left')

In [ ]:
naics_codes_17_22 

,naics_code,naics_title
0,11,"Agriculture, Forestry, Fishing and Hunting"
1,111,Crop Production
2,1111,Oilseed and Grain Farming
3,11111,Soybean Farming
4,111110,Soybean Farming
...,...,...
4316,9281,National Security and International Affairs
4317,92811,National Security
4318,928110,National Security
4319,92812,International Affairs


In [ ]:
df_sales_agg_by_naicsh_year

,year,naicsh,sales,naics_code,naics_title
0,2017,111310,129.829,111310,Orange Groves
1,2017,111335,35.657,111335,Tree Nut Farming
2,2017,111335,35.657,111335,Tree Nut Farming
3,2017,111998,0.0,111998,All Other Miscellaneous Crop Farming
4,2017,111998,0.0,111998,All Other Miscellaneous Crop Farming
...,...,...,...,...,...
6663,2026,519290,4887.0,519290,Web Search Portals and All Other Information S...
6664,2026,524114,1313.429,524114,Direct Health and Medical Insurance Carriers
6665,2026,524114,1313.429,524114,Direct Health and Medical Insurance Carriers
6666,2026,541512,7262.0,541512,Computer Systems Design Services


In [ ]:
naics_code_2022




,naics_code,naics_title
0,11,"Agriculture, Forestry, Fishing and Hunting"
1,111,Crop Production
2,1111,Oilseed and Grain Farming
3,11111,Soybean Farming
4,111110,Soybean Farming
...,...,...
2120,9281,National Security and International Affairs
2121,92811,National Security
2122,928110,National Security
2123,92812,International Affairs


,naics_code,naics_title
0,11,"Agriculture, Forestry, Fishing and Hunting"
1,111,Crop Production
2,1111,Oilseed and Grain Farming
3,11111,Soybean Farming
4,111110,Soybean Farming


In [ ]:
df_sales_agg_by_naicsh3_year

,year,naicsh_3digit,sales
0,2017,111,165.486
1,2017,113,99.823
2,2017,115,49.89
3,2017,211,91572.049
4,2017,212,13937.412
...,...,...,...
749,2026,513,1206.013
750,2026,518,18363.953
751,2026,519,4887.0
752,2026,524,1313.429
